In [24]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore", message="Could not infer format")
input_file = "10w bird data 2015-2026.csv"
output_file = "cleaned_bird_data.csv"
chunksize = 100000

if os.path.exists(output_file):
    os.remove(output_file)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

def preprocess(df_raw):
    # 1. Read Data
    df = df_raw.copy()

    # 2. Normalize Column Name
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_").str.replace("/", "_")

    # 3. Drop Duplicity
    df = df.drop_duplicates(keep="first")

    # 4. X = 0 & And "presence"
    if "observation_count" in df.columns:
        df["observation_count_raw"] = df["observation_count"].astype(str)
        df["observation_count"] = df["observation_count"].replace("X", np.nan)
        df["observation_count"] = pd.to_numeric(df["observation_count"], errors="coerce").fillna(0)
    df["presence"] = 1
 
    if "observation_count_raw" in df.columns:
        df.drop(columns=["observation_count_raw"], inplace=True)

    # Remove Other Languages and Special Signs in Locality
    if "locality" in df.columns:
        df["locality"] = df["locality"].astype(str)
    # 删除所有非中文、英文、数字、逗号空格的乱码符号
        df["locality"] = df["locality"].str.replace(r"[^\u4e00-\u9fa5a-zA-Z0-9, ]", "", regex=True)
    # 去除首尾空格、合并多个连续空格
        df["locality"] = df["locality"].str.strip().str.replace(r"\s+", " ", regex=True)
    # 清洗后为空的地名统一填充“未知地点”
        df["locality"] = df["locality"].replace("", "未知地点")
        
    # 5. Create: Year, Month, Day, Weekdan or Not, Day in a Year
    if "observation_date" in df.columns:
        df["observation_date"] = pd.to_datetime(df["observation_date"], errors="coerce")
        df["year"] = df["observation_date"].dt.year
        df["month"] = df["observation_date"].dt.month
        df["day"] = df["observation_date"].dt.day
        df["weekday"] = df["observation_date"].dt.weekday
        df["dayofyear"] = df["observation_date"].dt.dayofyear

    # 6. Observed Time 
    if "time_observations_started" in df.columns:
        df["time_observations_started"] = pd.to_datetime(df["time_observations_started"], errors="coerce")
        df["hour"] = df["time_observations_started"].dt.hour
    # Blank Space Insert "-1"
        df["hour"] = df["hour"].fillna(-1)

    # 7. Substitute Values
    num_cols = ["latitude", "longitude", "duration_minutes", "effort_distance_km", "effort_area_ha", "number_observers"]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 8. Other Added Columns
    if "weekday" in df.columns:
        df["is_weekend"] = (df["weekday"] >= 5).astype(int)
    if "month" in df.columns:
        df["migration_season"] = df["month"].isin([3,4,5,9,10,11]).astype(int)
    if "hour" in df.columns:
        df["morning_peak"] = df["hour"].between(5,9).astype(int)
        df["evening_peak"] = df["hour"].between(16,19).astype(int)

    # 9. Drop Columns
    drop_list = [
        "protocol_code",
        "effort_area_ha",
        "checklist_comments",
        "species_comments",
        "locality_type",
        "age_sex",
        "country_code", "state_code", "iba_code", "bcr_code", "usfws_code", "atlas_block",
        "exotic_code", "behavior_code", "breeding_code",
        "breeding_category",
        "subspecies_common_name", "subspecies_scientific_name",
        "global_unique_identifier", "taxon_concept_id", "locality_id",
        "sampling_event_identifier", "observer_id", "observer_orcid_id", "group_identifier",
        "project_names", "project_identifiers",
        "last_edited_date", "has_media", "approved", "reviewed"
    ]
    df = df.drop(columns=[c for c in drop_list if c in df.columns], errors="ignore")
    return df

# Iteration by Parts
first_header = True
total_read = 0
total_saved = 0

# Read csv and Insert function
for chunk in pd.read_csv(input_file, chunksize=chunksize, low_memory=False, encoding="utf-8"):
    total_read += len(chunk)
    clean_chunk = preprocess(chunk)
    total_saved += len(clean_chunk)

    # Cleaned Data
    clean_chunk.to_csv(
        output_file,
        mode="w" if first_header else "a",
        header=first_header,
        index=False,
        encoding="utf-8-sig"
    )
    first_header = False
    print(f"Readed Columns：{total_read} | Saved Columns：{total_saved}")

print("Finished Pre-Processing:")
print(f"Orginal Count：{total_read}")
print(f"Cleaned Count：{total_saved}")

final_df = pd.read_csv(output_file, low_memory=False)
print(final_df.columns.tolist())

Readed Columns：100000 | Saved Columns：100000
Readed Columns：200000 | Saved Columns：200000
Readed Columns：300000 | Saved Columns：300000
Readed Columns：400000 | Saved Columns：400000
Readed Columns：500000 | Saved Columns：500000
Readed Columns：600000 | Saved Columns：600000
Readed Columns：700000 | Saved Columns：700000
Readed Columns：800000 | Saved Columns：800000
Readed Columns：900000 | Saved Columns：900000
Readed Columns：1000000 | Saved Columns：1000000
Readed Columns：1048575 | Saved Columns：1048575
Finished Pre-Processing:
Orginal Count：1048575
Cleaned Count：1048575
['taxonomic_order', 'category', 'common_name', 'scientific_name', 'observation_count', 'country', 'state', 'county', 'county_code', 'locality', 'latitude', 'longitude', 'observation_date', 'time_observations_started', 'observation_type', 'protocol_name', 'duration_minutes', 'effort_distance_km', 'number_observers', 'all_species_reported', 'reason', 'presence', 'year', 'month', 'day', 'weekday', 'dayofyear', 'hour', 'is_weekend',

In [14]:
import os
print(os.getcwd())
print(os.path.abspath(output_file))

/Users/a20220927003
/Users/a20220927003/cleaned_bird_data.csv
